# 02. 멀티모달 숙제 인증 이미지 분석

이 notebook은 원본의 이미지 분석 실험을 보존합니다.

1. Gemma-3n-E2B-it 멀티모달 모델로 완료 여부와 완료율 분석
2. OpenCV 기반 채움 비율 및 원 검출 실험
3. OCR 기반 완료율 보조 판독

실제 학습자 이미지와 사설 S3 URL은 포함하지 않습니다. 로컬 테스트 이미지를 별도로 지정하세요.

In [ ]:
# %pip install -q -r ../requirements.txt

import os
import sys
from pathlib import Path

CODE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from dotenv import load_dotenv
load_dotenv(CODE_DIR / ".env")

IMAGE_PATH = None  # 예: "./private_images/test_homework.jpg"
ENABLE_GEMMA_ANALYSIS = False

## 1. 비식별 출석 데이터 형식

In [ ]:
import json

sample_path = CODE_DIR / "sample_data" / "sample_attendance.json"
attendance = json.loads(sample_path.read_text(encoding="utf-8"))
attendance

## 2. Gemma 멀티모달 분석

GPU와 Gemma 접근 권한이 있는 환경에서만 `ENABLE_GEMMA_ANALYSIS = True`로 변경하세요.

In [ ]:
from src.homework_image_analyzer import analyze_with_gemma, load_multimodal_gemma

if ENABLE_GEMMA_ANALYSIS and IMAGE_PATH:
    processor, mm_model = load_multimodal_gemma(hf_token=os.getenv("HF_TOKEN"))
    gemma_result = analyze_with_gemma(IMAGE_PATH, processor, mm_model)
    print(gemma_result)
else:
    print("Gemma 멀티모달 분석은 실행하지 않았습니다.")

## 3. OpenCV 기반 채움 비율·원 검출 실험

원본 구현에서 숙제 이미지의 검은색 픽셀 비율과 Hough Circle Transform으로 완료도를 보조적으로 추정했습니다. 이미지 양식에 민감하므로 실험적 접근으로 해석해야 합니다.

In [ ]:
from src.homework_image_analyzer import analyze_with_opencv

if IMAGE_PATH:
    opencv_result = analyze_with_opencv(IMAGE_PATH)
    print(opencv_result)
else:
    print("IMAGE_PATH를 설정하면 OpenCV 실험을 실행할 수 있습니다.")

## 4. OCR 기반 완료율 보조 판독

Gradio 프로토타입에서는 멀티모달 분석이 완료율을 반환하지 못했을 때 OCR을 보조적으로 사용했습니다.

In [ ]:
from src.homework_image_analyzer import analyze_with_ocr, analyze_with_fallbacks

if IMAGE_PATH:
    ocr_result = analyze_with_ocr(IMAGE_PATH)
    print(ocr_result)
    print(analyze_with_fallbacks(IMAGE_PATH))
else:
    print("IMAGE_PATH를 설정하면 OCR 및 fallback 흐름을 실행할 수 있습니다.")